# Rank Fusion (RRF) for Hybrid Search

Wiki reference for [Rank Fusion (RRF)](https://ml-viz-ruby.vercel.app/wiki/rank-fusion). To keep your own copy, use **File -> Save a copy in Drive**.

**The problem.** Hybrid search runs two retrievers on incomparable scales: BM25 keyword scores (~0-40) and cosine similarity (~0.6-0.9). If you *average the raw scores*, the larger-magnitude system (BM25) dominates and the fused list collapses onto the keyword order.

**The idea in one sentence.** Fuse *positions*, not scores: Reciprocal Rank Fusion gives each document a score of the form sum of 1/(k + rank_i), so only ranks matter and neither retriever can steamroll the other.

We build both fusion methods from scratch on a tiny toy corpus, reproduce the worked trace from the wiki, and visualize why averaging is absorption while RRF is negotiation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
})
np.random.seed(0)

# Eight documents scored by two retrievers on very different scales.
docs = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8']
bm25 = np.array([38, 33, 29, 25, 20, 16, 11, 7], dtype=float)   # keyword, 0-40
cos  = np.array([0.72, 0.68, 0.88, 0.65, 0.86, 0.63, 0.83, 0.61])  # cosine, 0.6-0.9

print('BM25 range :', bm25.min(), '-', bm25.max())
print('Cosine range:', cos.min(), '-', cos.max())

## 1 - The trap: averaging raw scores

Averaging a 38 with a 0.72 is averaging *40 rupees and 0.8 dollars*. The magnitude of BM25 swamps cosine, so the fused order is just the BM25 order.

In [ ]:
def order_from_scores(ids, scores):
    # returns document ids sorted by descending score, and a rank map (1-indexed)
    idx = np.argsort(-np.asarray(scores), kind='stable')
    ordered = [ids[i] for i in idx]
    rank = {d: r + 1 for r, d in enumerate(ordered)}
    return ordered, rank

bm25_order, bm25_rank = order_from_scores(docs, bm25)
dense_order, dense_rank = order_from_scores(docs, cos)

avg_scores = (bm25 + cos) / 2.0            # naive average of RAW scores
avg_order, _ = order_from_scores(docs, avg_scores)

print('BM25 order :', bm25_order)
print('Dense order:', dense_order)
print('Avg  order :', avg_order)
print('Averaging == BM25 order? ', avg_order == bm25_order)

The averaged order is **identical** to the BM25 order: the embedding retriever contributed nothing. This is *absorption* - the bigger currency ate the smaller one. Per-query min-max normalization can patch the scale but is brittle to a single outlier document.

## 2 - Reciprocal Rank Fusion, from scratch

$$\text{RRF}(d) = \sum_i \frac{1}{k + r_i(d)}$$

Only the ranks $r_i$ enter (1-indexed), so the score scales are irrelevant. The constant $k$ (default 60) discounts deep ranks: rank 1 -> 2 matters far more than rank 50 -> 51.

In [ ]:
def rrf(rank_maps, ids, k=60):
    scores = {}
    for d in ids:
        scores[d] = sum(1.0 / (k + rmap[d]) for rmap in rank_maps)
    return scores

rrf_scores = rrf([bm25_rank, dense_rank], docs, k=60)
rrf_order, rrf_rank = order_from_scores(docs, [rrf_scores[d] for d in docs])

for d in docs:
    print(f'{d}: bm25 #{bm25_rank[d]}  dense #{dense_rank[d]}  '
          f'rrf {rrf_scores[d]:.5f}  -> fused #{rrf_rank[d]}')
print()
print('RRF fused order:', rrf_order)

The embedding model's favourites (**D3, D5, D7** - buried by BM25) climb the fused list, while BM25's top docs are not thrown away. That reshuffling is exactly the value hybrid search is supposed to add.

### Reproducing the wiki's worked trace

Five documents, ranks given directly (not scores), to match the table on the wiki page.

In [ ]:
trace_ids = ['A', 'B', 'C', 'D', 'E']
bm25_r = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5}
dense_r = {'A': 4, 'B': 5, 'C': 1, 'D': 2, 'E': 3}

trace_scores = rrf([bm25_r, dense_r], trace_ids, k=60)
trace_order, _ = order_from_scores(trace_ids, [trace_scores[d] for d in trace_ids])
for d in trace_ids:
    print(f'{d}: {trace_scores[d]:.4f}')
print('Fused order:', trace_order)   # expected: C, A, D, B, E

## 3 - Visualize: absorption vs negotiation

A bump chart tracks each document's position across the three lists. Under **average** the fused column mirrors BM25; under **RRF** the lines cross as the dense favourites rise.

In [ ]:
favourites = {'D3', 'D5', 'D7'}

def bump_axis(ax, rank_cols, col_labels, title):
    n = len(docs)
    xs = np.arange(len(rank_cols))
    for d in docs:
        ys = [rank_cols[c][d] for c in range(len(rank_cols))]
        color = '#14b8a6' if d in favourites else '#475569'
        lw = 2.5 if d in favourites else 1.2
        ax.plot(xs, ys, '-o', color=color, lw=lw, ms=9, zorder=3 if d in favourites else 1)
        ax.text(xs[0] - 0.06, rank_cols[0][d], d, ha='right', va='center',
                fontsize=9, color=color)
        ax.text(xs[-1] + 0.06, rank_cols[-1][d], d, ha='left', va='center',
                fontsize=9, color=color)
    ax.set_xticks(xs); ax.set_xticklabels(col_labels)
    ax.set_yticks(range(1, n + 1))
    ax.invert_yaxis()
    ax.set_ylabel('rank (1 = best)')
    ax.set_title(title)

avg_rank = {d: r + 1 for r, d in enumerate(avg_order)}
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
bump_axis(axes[0], [bm25_rank, dense_rank, avg_rank],
          ['BM25', 'Dense', 'Average'], 'Averaging = absorption')
bump_axis(axes[1], [bm25_rank, dense_rank, rrf_rank],
          ['BM25', 'Dense', 'RRF'], 'RRF = negotiation')
plt.tight_layout(); plt.show()

**What to notice.** In the left panel the *Average* column is a copy of *BM25* - the teal favourite lines stay at the bottom. In the right panel the same favourites climb steeply: RRF let the dense retriever's votes count.

## 4 - Tradeoffs & when to use it

| | Average raw scores | Per-query min-max | **RRF** |
|---|---|---|---|
| Scale-invariant | No | Yes | **Yes** |
| Robust to outliers | No | No | **Yes** |
| Needs tuning per corpus | weights | - | just `k` (~60) |
| Uses score magnitude | Yes | Yes | **No (rank only)** |

- **Use RRF** as the default merge for any hybrid (sparse + dense) retrieval, or to combine 3+ ranked lists.
- **It is not a reranker** - it can't rescue a document both retrievers ranked poorly. Pair it with a cross-encoder reranker on the fused shortlist.
- **Missing documents:** if a doc appears in only one list, treat its rank in the other as infinity (contributes 0), don't drop it.

## 5 - Your turn

Implement RRF yourself and confirm it reproduces the worked trace. Fill in the `# TODO(you)` line.

In [ ]:
def my_rrf(rank_maps, ids, k=60):
    scores = {}
    for d in ids:
        # TODO(you): sum 1 / (k + rank) over every rank map for document d
        scores[d] = 0.0  # replace this
    return scores

_mine = my_rrf([bm25_r, dense_r], trace_ids, k=60)
_order, _ = order_from_scores(trace_ids, [_mine[d] for d in trace_ids])
assert _order == ['C', 'A', 'D', 'B', 'E'], f'got {_order}'
print('Correct! Fused order:', _order)

<details>
<summary>Solution</summary>

```python
def my_rrf(rank_maps, ids, k=60):
    scores = {}
    for d in ids:
        scores[d] = sum(1.0 / (k + rmap[d]) for rmap in rank_maps)
    return scores
```

The whole method is one line inside the loop: sum the reciprocal ranks. No score magnitudes appear anywhere - that is the entire point.
</details>

## Key takeaways

- **Averaging scores from different retrievers is absorption**: the larger-magnitude system dominates and the fused list becomes its list.
- **RRF fuses positions, not magnitudes** - `sum 1/(k + rank)` - so every retriever gets a vote and deep ranks are discounted.
- `k ~ 60` is a mild default; the ranking is insensitive to `k` over a wide range.
- RRF blends what you already retrieved; a **cross-encoder reranker** improves the *scoring* of the shortlist. They compose.

**Next:** [Retrieval-Augmented Generation](https://ml-viz-ruby.vercel.app/courses/building-with-llms/04-retrieval-augmented-generation) - where the fused list becomes the LLM's context - and [Vector Databases & ANN Indexes](https://ml-viz-ruby.vercel.app/wiki/vector-databases) for how the dense list is produced at scale.